# Scraper e classificação: notícias automotivas
Coleta Automotive Business e AutoData e classifica com o **prompt automotivo** (`ai_classify` relevância Alta/Média/Baixa, temas e empresa).
Grava em `hive_metastore.default.noticias_automotivebusiness_arcelormittal`, que o Export do portal lê.

**Só coleta e classificação.** O envio para o Teams saiu deste notebook (19/10/2026): as notícias automotivas aparecem no portal, e não no canal.
Roda pelo `00_atualizar_portal`.


In [0]:
%pip install requests beautifulsoup4 lxml

In [0]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import hashlib

# ID estável entre execuções (o hash() nativo do Python muda a cada sessão e gerava duplicatas no MERGE)
def stable_id(texto, digitos=15):
    return int(hashlib.md5(str(texto).encode("utf-8")).hexdigest(), 16) % (10 ** digitos)

BASE_URL = "https://www.automotivebusiness.com.br"
NOTICIAS_URL = f"{BASE_URL}/noticias"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": BASE_URL,
}

print("Configuração OK")

In [0]:
from html import unescape
import re as _re
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import socket

WP_API = "https://painel.automotivebusiness.com.br/wp-json/wp/v2"
FRONTEND = "www.automotivebusiness.com.br"
BACKEND  = "painel.automotivebusiness.com.br"

FIELDS = "id,date,title,link,excerpt,categories,tags,author,_links"

# --- Configuracao de Retry ---
MAX_RETRIES = 4          # tentativas totais (1 original + 3 retries)
BACKOFF_FACTOR = 2       # espera: 2s, 4s, 8s entre retries
RETRY_STATUS_CODES = [429, 500, 502, 503, 504]


def verificar_dns(host="painel.automotivebusiness.com.br"):
    """Verifica se o dominio resolve antes de iniciar o scraping."""
    try:
        socket.gethostbyname(host)
        return True
    except socket.gaierror:
        return False


def criar_session_com_retry():
    """Cria session HTTP com retry e backoff exponencial."""
    session = requests.Session()
    session.headers.update(HEADERS)
    retry_strategy = Retry(
        total=MAX_RETRIES,
        backoff_factor=BACKOFF_FACTOR,
        status_forcelist=RETRY_STATUS_CODES,
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def limpar_html(texto):
    """Remove tags HTML e decodifica entidades."""
    return unescape(_re.sub(r"<[^>]+>", "", texto or "")).strip()


def scrape_wp_api(n_paginas=5, por_pagina=100):
    """Coleta posts via WordPress REST API com paginacao e retry."""

    # Pre-check: verificar DNS antes de tentar requests
    print("Verificando conectividade DNS...", end=" ")
    if not verificar_dns():
        print("❌ FALHOU")
        print("⚠️  DNS nao resolve 'painel.automotivebusiness.com.br'.")
        print("   Possivel bloqueio de rede/firewall na VNet.")
        print("   Scraping abortado graciosamente — pipeline continua sem novos dados.")
        return []  # Retorna vazio em vez de crashar o job

    print("✅ OK")
    session = criar_session_com_retry()
    todos = []

    for pagina in range(1, n_paginas + 1):
        params = {
            "page"     : pagina,
            "per_page" : por_pagina,
            "_fields"  : FIELDS,
            "orderby"  : "date",
            "order"    : "desc",
        }
        try:
            resp = session.get(f"{WP_API}/posts", params=params, timeout=30)
        except requests.exceptions.ConnectionError as e:
            print(f"\n❌ Erro de conexao na pagina {pagina}: {e}")
            print("   Retornando dados coletados ate agora.")
            break
        except requests.exceptions.Timeout:
            print(f"\n⏰ Timeout na pagina {pagina} (30s). Retornando dados parciais.")
            break
        except requests.exceptions.RequestException as e:
            print(f"\n❌ Erro inesperado na pagina {pagina}: {e}")
            break

        print(f"Pagina {pagina} -> HTTP {resp.status_code}", end="")

        if resp.status_code == 400:
            print(" (sem mais paginas)")
            break
        if resp.status_code != 200:
            print(f" ERRO: {resp.text[:200]}")
            break

        posts = resp.json()
        print(f" | {len(posts)} posts")
        if not posts:
            break

        total_pages = int(resp.headers.get("X-WP-TotalPages", 1))

        for p in posts:
            link_frontend = p.get("link", "").replace(BACKEND, FRONTEND)
            todos.append({
                "id"         : p.get("id"),
                "data"       : p.get("date"),
                "titulo"     : limpar_html(p.get("title",   {}).get("rendered", "")),
                "resumo"     : limpar_html(p.get("excerpt", {}).get("rendered", "")),
                "link"       : link_frontend,
                "categorias" : str(p.get("categories", [])),
                "tags"       : str(p.get("tags", [])),
                "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        if pagina >= total_pages:
            print(f"Total de paginas: {total_pages} — coleta concluida.")
            break

        time.sleep(0.5)

    return todos


# --- Execucao ---
# Para coleta diaria incremental: 2 paginas (200 artigos mais recentes) e suficiente
# Para reprocessamento completo do historico, aumente para n_paginas=20 ou mais
noticias = scrape_wp_api(n_paginas=2, por_pagina=100)
df_noticias = pd.DataFrame(noticias)
print(f"\nArtigos coletados nesta execucao: {len(df_noticias)}")
if not df_noticias.empty:
    display(df_noticias.head(5))
else:
    print("Nenhum artigo coletado — verificar conectividade de rede.")

In [0]:
from html import unescape
import re as _re
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import socket

# ============================================================
# SCRAPER: www.autodata.com.br
# Estratégia: WordPress REST API (principal) + HTML fallback
# ============================================================

AUTODATA_DOMAIN = "www.autodata.com.br"
AUTODATA_WP_API = f"https://{AUTODATA_DOMAIN}/wp-json/wp/v2"
AUTODATA_URL = f"https://{AUTODATA_DOMAIN}"


def verificar_dns_autodata():
    """Verifica se o domínio resolve antes de iniciar o scraping."""
    try:
        socket.gethostbyname(AUTODATA_DOMAIN)
        return True
    except socket.gaierror:
        return False


def criar_session_autodata():
    """Cria session HTTP com retry e backoff exponencial."""
    session = requests.Session()
    session.headers.update(HEADERS)
    retry_strategy = Retry(
        total=4,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def scrape_autodata_wp_api(session, n_paginas=2, por_pagina=100):
    """Tenta coletar via WordPress REST API."""
    todos = []
    fields = "id,date,title,link,excerpt,categories,tags"

    for pagina in range(1, n_paginas + 1):
        params = {
            "page": pagina,
            "per_page": por_pagina,
            "_fields": fields,
            "orderby": "date",
            "order": "desc",
        }
        try:
            resp = session.get(f"{AUTODATA_WP_API}/posts", params=params, timeout=30)
        except Exception as e:
            print(f"  ❌ Erro na pagina {pagina}: {e}")
            break

        if resp.status_code == 404:
            print(f"  WP API não disponível (404). Usando fallback HTML.")
            return None  # Sinaliza para usar fallback
        if resp.status_code == 400:
            print(f"  Pagina {pagina}: sem mais resultados.")
            break
        if resp.status_code != 200:
            print(f"  Pagina {pagina}: HTTP {resp.status_code} - {resp.text[:200]}")
            break

        posts = resp.json()
        print(f"  Pagina {pagina} -> {len(posts)} posts")
        if not posts:
            break

        for p in posts:
            todos.append({
                "id": p.get("id"),
                "data": p.get("date"),
                "titulo": limpar_html(p.get("title", {}).get("rendered", "")),
                "resumo": limpar_html(p.get("excerpt", {}).get("rendered", "")),
                "link": p.get("link", ""),
                "categorias": str(p.get("categories", [])),
                "tags": str(p.get("tags", [])),
                "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        time.sleep(0.5)

    return todos


def scrape_autodata_html(session, n_paginas=5):
    """Fallback: coleta via scraping HTML das páginas de notícias."""
    todos = []
    
    for pagina in range(1, n_paginas + 1):
        url = f"{AUTODATA_URL}/noticias/page/{pagina}/" if pagina > 1 else f"{AUTODATA_URL}/noticias/"
        try:
            resp = session.get(url, timeout=30)
        except Exception as e:
            print(f"  ❌ Erro HTML pagina {pagina}: {e}")
            break

        if resp.status_code != 200:
            print(f"  Pagina {pagina}: HTTP {resp.status_code}")
            break

        soup = BeautifulSoup(resp.text, "lxml")
        
        # Tentar encontrar artigos - padrões comuns WordPress
        artigos = soup.find_all("article") or soup.select(".post, .entry, .news-item, .td-module-thumb")
        
        if not artigos:
            # Fallback: buscar links com padrão de notícia
            artigos = soup.select("h2 a, h3 a, .entry-title a")
            if not artigos:
                print(f"  Pagina {pagina}: nenhum artigo encontrado no HTML.")
                break
            
            for a in artigos:
                link = a.get("href", "")
                if not link.startswith("http"):
                    link = AUTODATA_URL + link
                titulo = a.get_text(strip=True)
                if titulo and len(titulo) > 15:  # Filtrar links curtos/menus
                    todos.append({
                        "id": stable_id(link, 9),  # ID único baseado no link
                        "data": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
                        "titulo": titulo,
                        "resumo": "",
                        "link": link,
                        "categorias": "[]",
                        "tags": "[]",
                        "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    })
        else:
            for art in artigos:
                # Extrair título
                titulo_tag = art.find(["h2", "h3", "h4"]) or art.find(class_=_re.compile(r"title"))
                titulo = titulo_tag.get_text(strip=True) if titulo_tag else ""
                
                # Extrair link
                link_tag = art.find("a", href=True) if not art.name == "a" else art
                link = link_tag["href"] if link_tag else ""
                if link and not link.startswith("http"):
                    link = AUTODATA_URL + link
                
                # Extrair resumo
                resumo_tag = art.find(class_=_re.compile(r"excerpt|summary|desc")) or art.find("p")
                resumo = resumo_tag.get_text(strip=True) if resumo_tag else ""
                
                # Extrair data
                data_tag = art.find("time") or art.find(class_=_re.compile(r"date|time"))
                data_str = data_tag.get("datetime", data_tag.get_text(strip=True)) if data_tag else datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
                
                if titulo and len(titulo) > 15:
                    todos.append({
                        "id": stable_id(link, 9),
                        "data": data_str,
                        "titulo": limpar_html(titulo),
                        "resumo": limpar_html(resumo),
                        "link": link,
                        "categorias": "[]",
                        "tags": "[]",
                        "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    })

        print(f"  Pagina {pagina} -> {len(artigos)} artigos encontrados")
        time.sleep(1)  # Mais conservador para HTML scraping

    return todos


# --- Execução Autodata ---
print("="*60)
print("SCRAPER: www.autodata.com.br")
print("="*60)

print("\nVerificando DNS...", end=" ")
if not verificar_dns_autodata():
    print("❌ FALHOU")
    print("⚠️  DNS não resolve 'www.autodata.com.br'.")
    print("   Possível bloqueio de rede/firewall na VNet.")
    print("   Scraping Autodata abortado — pipeline continua sem esta fonte.")
    df_noticias_autodata = pd.DataFrame()
else:
    print("✅ OK")
    session_ad = criar_session_autodata()
    
    # Estratégia 1: WordPress REST API
    print("\n📡 Tentando WordPress REST API...")
    noticias_autodata = scrape_autodata_wp_api(session_ad, n_paginas=2, por_pagina=100)
    
    # Estratégia 2: Fallback HTML
    if noticias_autodata is None:
        print("\n🔄 Usando fallback HTML scraping...")
        noticias_autodata = scrape_autodata_html(session_ad, n_paginas=5)
    
    df_noticias_autodata = pd.DataFrame(noticias_autodata) if noticias_autodata else pd.DataFrame()
    print(f"\n✅ Autodata: {len(df_noticias_autodata)} artigos coletados")
    if not df_noticias_autodata.empty:
        display(df_noticias_autodata.head(5))

In [0]:
# Combinar fontes: AutomotiveBusiness + Autodata
# (o volume de keywords do LinkedIn saiu daqui: são posts de siderúrgicas, que têm o prompt próprio do pipeline Infomet/LinkedIn)
_fontes = []
if not df_noticias.empty:
    df_noticias["fonte_site"] = "automotivebusiness"
    _fontes.append(df_noticias)
    print(f"  AutomotiveBusiness: {len(df_noticias)} artigos")

if 'df_noticias_autodata' in dir() and not df_noticias_autodata.empty:
    df_noticias_autodata["fonte_site"] = "autodata"
    # Garantir schema compativel
    for col in ["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]:
        if col not in df_noticias_autodata.columns:
            df_noticias_autodata[col] = ""
    _fontes.append(df_noticias_autodata[["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]])
    print(f"  Autodata:           {len(df_noticias_autodata)} artigos")

if 'df_noticias_keywords' in dir() and not df_noticias_keywords.empty:
    for col in ["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]:
        if col not in df_noticias_keywords.columns:
            df_noticias_keywords[col] = ""
    _fontes.append(df_noticias_keywords[["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]])
    print(f"  Keywords (Volume):  {len(df_noticias_keywords)} registros")

if _fontes:
    df_todas_noticias = pd.concat(_fontes, ignore_index=True)
    print(f"  TOTAL:              {len(df_todas_noticias)} artigos para MERGE")
    spark_df = spark.createDataFrame(df_todas_noticias)
    spark_df = spark_df.dropDuplicates(["id", "fonte_site"])
    spark_df.createOrReplaceTempView("_noticias_novas")

    # Cria tabela Delta se nao existir (com coluna fonte_site)
    spark.sql("""
        CREATE TABLE IF NOT EXISTS noticias_automotivebusiness (
            id         LONG,
            data       STRING,
            titulo     STRING,
            resumo     STRING,
            link       STRING,
            categorias STRING,
            tags       STRING,
            coletado_em STRING,
            fonte_site STRING COMMENT 'Fonte: automotivebusiness ou autodata'
        ) USING DELTA
    """)

    # Adicionar coluna fonte_site caso tabela ja exista sem ela
    try:
        spark.sql("ALTER TABLE noticias_automotivebusiness ADD COLUMN fonte_site STRING COMMENT 'Fonte: automotivebusiness ou autodata'")
        print("  + coluna 'fonte_site' adicionada.")
    except Exception as e:
        if "already exists" in str(e).lower():
            pass
        else:
            raise

    # Backfill: marcar registros antigos sem fonte como 'automotivebusiness'
    spark.sql("""
        UPDATE noticias_automotivebusiness
        SET fonte_site = 'automotivebusiness'
        WHERE fonte_site IS NULL
    """)

    # MERGE incremental: chave composta (id + fonte_site) para evitar conflito entre fontes
    result = spark.sql("""
        MERGE INTO noticias_automotivebusiness t
        USING _noticias_novas s ON t.id = s.id AND t.fonte_site = s.fonte_site
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    total = spark.sql("SELECT COUNT(*) AS n FROM noticias_automotivebusiness").collect()[0][0]
    print(f"\nMERGE concluido. Total na tabela: {total} artigos.")
else:
    print("Nenhuma fonte retornou dados — nada foi salvo.")

In [0]:
# Adiciona novas colunas ao modelo de dados (padrao pipeline Infomet)
# Safe: usa try/except para cada coluna individualmente
nova_cols = [
    ("score_relevancia",    "INT"),
    ("empresa_principal",   "STRING"),
    ("categoria_principal", "STRING"),
]

for nome, tipo in nova_cols:
    try:
        spark.sql(f"""
            ALTER TABLE hive_metastore.default.noticias_automotivebusiness_arcelormittal
            ADD COLUMN {nome} {tipo}
        """)
        print(f"  + coluna '{nome}' adicionada.")
    except Exception as e:
        msg = str(e).lower()
        if "already exists" in msg:
            print(f"  ~ coluna '{nome}' ja existe.")
        elif "table_or_view_not_found" in msg or "does not exist" in msg:
            print(f"  ! tabela ainda nao existe - sera criada com todas as colunas na celula de classificacao.")
            break
        else:
            raise

# Backfill score para artigos ja classificados (sem score)
try:
    spark.sql("""
        UPDATE hive_metastore.default.noticias_automotivebusiness_arcelormittal
        SET score_relevancia = CASE relevancia
            WHEN 'Alta'  THEN 9
            WHEN 'M\u00e9dia' THEN 5
            WHEN 'Baixa' THEN 2
            ELSE 0
        END
        WHERE score_relevancia IS NULL
    """)
    # Backfill categoria_principal
    spark.sql("""
        UPDATE hive_metastore.default.noticias_automotivebusiness_arcelormittal
        SET categoria_principal = get(from_json(temas, 'ARRAY<STRING>'), 0)
        WHERE categoria_principal IS NULL AND temas IS NOT NULL
    """)
    n = spark.sql("""
        SELECT COUNT(*) FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal
        WHERE score_relevancia IS NOT NULL
    """).collect()[0][0]
    print(f"\nBackfill concluido: {n} artigos com score preenchido.")
except Exception as e:
    if "table_or_view_not_found" in str(e).lower() or "does not exist" in str(e).lower():
        print("Tabela classificada ainda nao existe - backfill sera feito apos a classificacao.")
    else:
        raise

In [0]:
%sql
CREATE TABLE IF NOT EXISTS noticias_automotivebusiness_arcelormittal (
  id BIGINT,
  data STRING,
  titulo STRING,
  resumo STRING,
  link STRING,
  categorias STRING,
  tags STRING,
  coletado_em STRING,
  fonte_site STRING COMMENT 'Fonte: automotivebusiness ou autodata',
  relevancia STRING,
  temas STRING,
  erro_relevancia     STRING,
  score_relevancia    INT,
  empresa_principal   STRING,
  categoria_principal STRING
) USING DELTA;

-- coluna fonte_site ja incluida no CREATE TABLE acima

INSERT INTO noticias_automotivebusiness_arcelormittal
WITH base AS (
  SELECT
    s.id, s.data, s.titulo, s.resumo, s.link, s.categorias, s.tags, s.coletado_em, s.fonte_site,
    concat(s.titulo, '. ', COALESCE(s.resumo, '')) AS texto_completo
  FROM noticias_automotivebusiness s
  LEFT JOIN noticias_automotivebusiness_arcelormittal t
    ON s.id = t.id
  WHERE t.id IS NULL
),
classificadas AS (
  SELECT
    *,
    ai_classify(
      texto_completo,
      '{
         "Alta": "Notícias que afetam diretamente o volume de produção física, o mix de materiais ou a cadeia de fornecimento direto de aços planos e longos no ecossistema automotivo brasileiro. Inclui obrigatoriamente: Mudanças em volumes de produção, projeções da ANFAVEA e dados mensais de emplacamentos de marcas de grande volume (Fiat, VW, GM, Toyota, Hyundai, BYD, Stellantis). Anúncios de investimentos em novas linhas de montagem, novas fábricas ou expansão de capacidade produtiva no Brasil. Alterações na engenharia de materiais: substituição de aço por alumínio/plásticos, adoção de aços avançados de alta resistência (AHSS), ou aços específicos para segurança/baterias de veículos elétricos e híbridos. Dinâmicas severas na cadeia de autopeças e estampagem que usem aço como matéria-prima. Programas governamentais focados em incentivos diretos à produção local (como o Programa Mover) ou barreiras tarifárias contra o aço/veículos importados.",

          "Média": "Notícias que trazem tendências de mercado de médio/longo prazo ou movimentos institucionais que não alteram a produção de aço no curtíssimo prazo, mas desenham o futuro do setor. Inclui obrigatoriamente: Lançamentos de novos modelos de montadoras de grande volume (foco no produto/mercado, sem anúncio de novas fábricas). Discussões institucionais e regulatórias sobre ESG na cadeia automotiva, metas de descarbonização de escopo 3 e créditos de carbono. Movimentações de comércio exterior automotivo geral (importação/exportação de veículos acabados que não firam diretamente a produção local). Mudanças executivas no alto escalão das montadoras ou fusões globais sem impacto anunciado nas plantas brasileiras. EXCEÇÃO: Se a notícia de ESG ou descarbonização mencionar exigência de aço verde ou aço de baixa pegada de carbono do fornecedor, classificar como Alta.",

          "Baixa": "Notícias institucionais, mercadológicas superficiais, eventos isolados ou focadas em mercados de nicho sem escala industrial de consumo de aço. Inclui obrigatoriamente: Premiações do setor (Ex: O Carro do Ano), feiras de negócios regionais, patrocínios e campanhas de marketing de montadoras. Mercado de luxo, superesportivos ou veículos de nicho com volumes irrelevantes de emplacamento (Ferrari, Porsche, marcas de importação independente). Curiosidades históricas do setor automotivo, notícias sobre trânsito, comportamento do consumidor no varejo concessionário ou reajustes pontuais de tabela de preço ao consumidor final."
      }',
      MAP(
        'version','2.0',
        'instructions','Classifique cada notícia sob a ótica da ArcelorMittal Brasil como fornecedora de aços planos e longos para a indústria automotiva. Se uma notícia se enquadrar simultaneamente em critérios de duas categorias, priorize sempre a classificação mais alta. Em caso de dúvida sobre impacto em volume de aço, assuma o viés do fornecedor siderúrgico e classifique de forma conservadora (tendendo a subir a relevância).'
      )
    ) AS relevancia_variant,
    ai_classify(
      texto_completo,
      '{
        "Eletrificação": "Veículos elétricos, híbridos, plug-in, hidrogênio, baterias, motor elétrico.",
        "Materiais e Lightweighting": "Aço, alumínio, compósitos, materiais leves, chapas, redução de peso.",
        "Volume e Mercado": "Emplacamentos, vendas, produção, demanda, market share.",
        "Cadeia de Fornecedores": "Fornecedores, autopeças, componentes, nacionalização, supply chain.",
        "Regulação e Política Industrial": "Governo, impostos, tarifas, Mover, Rota 2030, regulação.",
        "Grandes Montadoras": "VW, GM, Stellantis, Ford, Toyota, Hyundai, Renault, Fiat, Honda, BYD, GWM.",
        "Sustentabilidade e ESG": "Descarbonização, emissões, sustentabilidade, reciclagem.",
        "Eventos e Entretenimento": "Prêmios, eventos, feiras, motorsport, patrocínios."
      }',
      MAP(
        'version','2.0',
        'multilabel','true',
        'instructions','Identifique todos os temas estratégicos presentes na notícia sob a ótica da ArcelorMittal Brasil.'
      )
    ) AS temas_variant,
    ai_extract(
      texto_completo,
      '{"empresa_principal": {"type": "string", "description": "Nome curto da principal montadora ou empresa automotiva mencionada. Use apenas o nome comercial curto: Volkswagen, Toyota, BYD, Honda, Renault, Fiat, Chevrolet, Ford, Hyundai, Stellantis, GWM, Mercedes-Benz, Nissan, Volvo, BMW, Audi, Kia. Retornar null se nenhuma empresa específica for o foco."}}',
      MAP('version', '2.0')
    ) AS empresa_variant
  FROM base
)
SELECT
  id,
  data,
  titulo,
  resumo,
  link,
  categorias,
  tags,
  coletado_em,
  fonte_site,
  relevancia_variant:response[0]::STRING                    AS relevancia,
  to_json(temas_variant:response)                           AS temas,
  relevancia_variant:error_message::STRING                  AS erro_relevancia,
  CASE relevancia_variant:response[0]::STRING
    WHEN 'Alta'  THEN 9
    WHEN 'Média' THEN 5
    WHEN 'Baixa' THEN 2
    ELSE 0
  END                                                       AS score_relevancia,
  empresa_variant:empresa_principal::STRING                 AS empresa_principal,
  temas_variant:response[0]::STRING                        AS categoria_principal
FROM classificadas

In [0]:
%sql
-- Resumo da rodada: última notícia coletada, última classificação e volume por relevância nos últimos 7 dias
SELECT fonte_site,
       MAX(DATE(to_timestamp(data)))        AS ultima_publicacao,
       MAX(DATE(to_timestamp(coletado_em))) AS ultima_coleta,
       SUM(CASE WHEN relevancia = 'Alta'  AND DATE(to_timestamp(data)) >= current_date() - INTERVAL 7 DAYS THEN 1 ELSE 0 END) AS alta_7d,
       SUM(CASE WHEN relevancia = 'Média' AND DATE(to_timestamp(data)) >= current_date() - INTERVAL 7 DAYS THEN 1 ELSE 0 END) AS media_7d,
       SUM(CASE WHEN relevancia = 'Baixa' AND DATE(to_timestamp(data)) >= current_date() - INTERVAL 7 DAYS THEN 1 ELSE 0 END) AS baixa_7d,
       SUM(CASE WHEN erro_relevancia IS NOT NULL THEN 1 ELSE 0 END) AS com_erro
FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal
GROUP BY fonte_site
ORDER BY fonte_site
